# BayMorph — Fit an Observed Single-Band Image

This notebook executes the reviewed analysis defined by
`02A_Observed_Assessment_and_Config.ipynb`.

The YAML is the reproducible contract between assessment and inference.
It records the chosen sampler and can be used from this notebook, the
BayMorph CLI, or a remote compute system.


In [ ]:
from pathlib import Path

import yaml

from baymorph import BayMorphRunner


## 1. Run the all-in-one workflow

Select the YAML reviewed and written by notebook 02A. Relative paths are
resolved from the BayMorph project root. The cell prints the resolved
sampler before starting the expensive work.

For emcee, `checkpoint_mode: fresh` refuses to overwrite an existing
checkpoint. To continue a suitable chain, review the YAML, change the
mode to `resume`, and increase `max_steps`; this is the total requested
production length, not an increment. A checkpoint already marked
`converged` adds no further steps.


In [ ]:
config_path = Path("configs/observed_galaxy.yaml")

runner = BayMorphRunner(config_path)
sampler_config = runner.config["fit"]["sampler"]

print("Resolved sampler configuration:")
print(
    yaml.safe_dump(
        sampler_config,
        sort_keys=False,
    )
)
if sampler_config["name"] == "emcee":
    print(
        "emcee is posterior-only; this run will not produce logZ."
    )

fit = runner.run(diagnostics=True)


### Step-by-step equivalent

The all-in-one call is recommended for normal use. To inspect the stages
explicitly in a fresh Python session:

    runner = BayMorphRunner(config_path)
    runner.prepare_image()
    runner.select_psf_candidates()
    runner.build_psf()
    fit = runner.fit()
    runner.diagnostics()

Do not execute both workflows for the same fresh emcee checkpoint.


## 2. Inspect the result

Every backend supplies posterior samples, provenance, goodness-of-fit
statistics, best-fitting parameters, prior-boundary diagnostics, and
persisted products. Evidence is displayed only when the selected backend
estimates it.

For emcee, inspect trace stability, autocorrelation estimates, MCMC
effective sample size, acceptance fractions, agreement across independent
seeds, residuals, and prior-boundary diagnostics. Independent seeds require
distinct analysis names and fresh checkpoints. A checkpoint is a
continuation mechanism, not evidence of convergence. Conservative
compatibility checks may also reject resume after BayMorph source or
scientific-problem changes.


In [ ]:
inference = fit.posterior
fit_summary = fit.summary()

if inference.evidence is None:
    print("Evidence: unavailable (posterior-only backend)")
    print(
        "MCMC effective sample size:",
        inference.effective_sample_size,
    )
    print(
        "Converged:",
        inference.diagnostics.get("converged"),
    )
else:
    print(f"logZ: {inference.logz}")
    print(f"logZ uncertainty: {inference.logz_error}")

fit_summary
